In [1]:
# =============================================================================
# 08_ECL_CALCULATION.IPYNB - COPY/PASTE NOTEBOOK CELLS
# Banking Stress Testing Framework - Python V2 Migration
#
# SAS V1 GOLDEN-REFERENCE RULES
# EAD = Outstanding_Balance
# Base_ECL = EAD * Base_PD * Base_LGD
# Stressed_ECL = EAD * Stressed_PD * Stressed_LGD
# Incremental_Stress_Loss = Stressed_ECL - Base_ECL
#
# INSTRUCTIONS
# 1. Create 08_ECL_Calculation.ipynb in Colab.
# 2. Create ten separate Code cells.
# 3. Copy each CELL section below into its corresponding Colab cell.
# 4. Run CELL 01 through CELL 10 strictly in sequence.
# 5. Continue only after BLOCK STATUS : SUCCESS.
# =============================================================================

# =============================================================================
# CELL 01 - MODULE HEADER
# =============================================================================

from datetime import datetime

module_start_timestamp = datetime.now()

print("=" * 80)
print("BANKING STRESS TESTING FRAMEWORK")
print("PYTHON V2 MIGRATION")
print("MODULE : 08_ECL_CALCULATION")
print("EXECUTION STARTED :", module_start_timestamp.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)



BANKING STRESS TESTING FRAMEWORK
PYTHON V2 MIGRATION
MODULE : 08_ECL_CALCULATION
EXECUTION STARTED : 2026-10-13 16:04:16


In [2]:
# =============================================================================
# CELL 02 - BLOCK 1: ENVIRONMENT, PATHS, AND LOGGING
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    from google.colab import drive
    from pathlib import Path
    import logging
    import sys
    import numpy as np
    import pandas as pd

    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/drive/MyDrive/Banking_Stress_Testing_V2_Python")
    INPUT_FILE = PROJECT_ROOT / "output" / "portfolio_lgd.csv"
    OUTPUT_FILE = PROJECT_ROOT / "output" / "portfolio_ecl.csv"
    VALIDATION_DIR = PROJECT_ROOT / "validation"
    LOG_DIR = PROJECT_ROOT / "logs"

    SUMMARY_FILE = VALIDATION_DIR / "08_ecl_calculation_summary.csv"
    VALIDATION_FILE = VALIDATION_DIR / "08_ecl_calculation_validation.csv"
    SCENARIO_SUMMARY_FILE = VALIDATION_DIR / "08_ecl_scenario_summary.csv"
    RISK_SUMMARY_FILE = VALIDATION_DIR / "08_ecl_risk_segment_summary.csv"
    PRODUCT_SUMMARY_FILE = VALIDATION_DIR / "08_ecl_product_summary.csv"
    TOP_ACCOUNTS_FILE = VALIDATION_DIR / "08_top_25_stressed_accounts.csv"
    SAMPLE_FILE = VALIDATION_DIR / "08_ecl_calculation_sample.csv"
    LOG_FILE = LOG_DIR / "08_ECL_Calculation.log"

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("08_ECL_Calculation")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    file_handler = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
    file_handler.setFormatter(formatter)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(f"Project root was not found: {PROJECT_ROOT}")
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Module 07 output was not found: {INPUT_FILE}")

    print("Project Root Exists:", PROJECT_ROOT.exists())
    print("Module 07 Output Exists:", INPUT_FILE.exists())
    print("ECL Input:", INPUT_FILE)
    print("ECL Output:", OUTPUT_FILE)
    print("Validation Directory:", VALIDATION_DIR)
    print("Execution Log:", LOG_FILE)

    logger.info("Environment, paths, and logging configured successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    if "logger" in locals():
        logger.exception("Block 1 failed during environment configuration.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 08: ECL CALCULATION
BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING
START TIME : 2026-10-13 16:07:20
Mounted at /content/drive
Project Root Exists: True
Module 07 Output Exists: True
ECL Input: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_lgd.csv
ECL Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_ecl.csv
Validation Directory: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/08_ECL_Calculation.log
2026-10-13 16:08:01,379 | INFO | Environment, paths, and logging configured successfully.


INFO:08_ECL_Calculation:Environment, paths, and logging configured successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 0
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 0
END TIME           : 2026-10-13 16:08:01
DURATION (SECONDS) : 40.88


In [3]:
# =============================================================================
# CELL 03 - BLOCK 2: LOAD AND VALIDATE MODULE 07 OUTPUT
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 2 : LOAD AND VALIDATE MODULE 07 OUTPUT")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    required_columns = [
        "Customer_ID", "Loan_ID", "Scenario_ID", "Scenario_Name",
        "Risk_Segment", "Product_Type", "Outstanding_Balance",
        "Base_PD", "Stressed_PD", "Base_LGD", "Stressed_LGD"
    ]

    portfolio_lgd = pd.read_csv(INPUT_FILE, low_memory=False)
    portfolio_ecl = portfolio_lgd.copy(deep=True)

    input_record_count = len(portfolio_ecl)
    input_column_count = len(portfolio_ecl.columns)
    missing_required_columns = sorted(set(required_columns) - set(portfolio_ecl.columns))
    duplicate_keys = int(portfolio_ecl[["Loan_ID", "Scenario_ID"]].duplicated().sum())
    missing_required_values = int(portfolio_ecl[required_columns].isna().sum().sum())
    invalid_outstanding_balance = int((portfolio_ecl["Outstanding_Balance"] < 0).sum())

    if missing_required_columns:
        raise ValueError(f"Required input columns are missing: {missing_required_columns}")
    if duplicate_keys > 0:
        raise ValueError("Duplicate Loan_ID and Scenario_ID combinations were found.")
    if missing_required_values > 0:
        raise ValueError("Missing values were found in required ECL input fields.")
    if invalid_outstanding_balance > 0:
        raise ValueError("Negative Outstanding_Balance values were found.")
    if input_record_count != 30000:
        raise ValueError("Module 07 input does not contain the expected 30,000 records.")

    numeric_columns = [
        "Outstanding_Balance", "Base_PD", "Stressed_PD", "Base_LGD", "Stressed_LGD"
    ]
    for column in numeric_columns:
        portfolio_ecl[column] = pd.to_numeric(portfolio_ecl[column], errors="raise")

    records_processed = input_record_count

    print("=" * 80)
    print("MODULE 07 OUTPUT LOADED SUCCESSFULLY")
    print("=" * 80)
    print("Input Records:", input_record_count)
    print("Input Columns:", input_column_count)
    print("Missing Required Columns:", missing_required_columns)
    print("Duplicate Loan-Scenario Keys:", duplicate_keys)
    print("Missing Required Input Values:", missing_required_values)
    print("Negative Outstanding Balances:", invalid_outstanding_balance)
    print("Input Validation Status: PASS")

    display(portfolio_ecl[[
        "Customer_ID", "Loan_ID", "Scenario_Name", "Risk_Segment",
        "Product_Type", "Outstanding_Balance", "Stressed_PD", "Stressed_LGD"
    ]].head(15))

    logger.info(
        "Module 07 output loaded successfully: %s records and %s columns.",
        input_record_count, input_column_count
    )
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 2 failed while loading Module 07 output.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)



MODULE 08: ECL CALCULATION
BLOCK 2 : LOAD AND VALIDATE MODULE 07 OUTPUT
START TIME : 2026-10-13 16:09:25
MODULE 07 OUTPUT LOADED SUCCESSFULLY
Input Records: 30000
Input Columns: 38
Missing Required Columns: []
Duplicate Loan-Scenario Keys: 0
Missing Required Input Values: 0
Negative Outstanding Balances: 0
Input Validation Status: PASS


,Customer_ID,Loan_ID,Scenario_Name,Risk_Segment,Product_Type,Outstanding_Balance,Stressed_PD,Stressed_LGD
0,1000001,50000001,Baseline,Low Risk,Personal Loan,1733362.19,0.0200,0.6000
1,1000001,50000001,Mild_Stress,Low Risk,Personal Loan,1733362.19,0.0250,0.6600
2,1000001,50000001,Severe_Stress,Low Risk,Personal Loan,1733362.19,0.0350,0.7500
3,1000002,50000002,Baseline,Low Risk,Corporate Loan,62244073.93,0.0200,0.4500
4,1000002,50000002,Mild_Stress,Low Risk,Corporate Loan,62244073.93,0.0250,0.4950
5,1000002,50000002,Severe_Stress,Low Risk,Corporate Loan,62244073.93,0.0350,0.5625
6,1000003,50000003,Baseline,High Risk,Credit Card,303259.15,0.1000,0.8500
7,1000003,50000003,Mild_Stress,High Risk,Credit Card,303259.15,0.1250,0.9350
8,1000003,50000003,Severe_Stress,High Risk,Credit Card,303259.15,0.1750,1.0000
9,1000004,50000004,Baseline,Medium Risk,Auto Loan,993246.23,0.0500,0.3500


2026-10-13 16:09:26,740 | INFO | Module 07 output loaded successfully: 30000 records and 38 columns.


INFO:08_ECL_Calculation:Module 07 output loaded successfully: 30000 records and 38 columns.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:09:26
DURATION (SECONDS) : 1.209


In [4]:
# =============================================================================
# CELL 04 - BLOCK 3: DERIVE EXPOSURE AT DEFAULT
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 3 : DERIVE EXPOSURE AT DEFAULT")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    portfolio_ecl["EAD"] = portfolio_ecl["Outstanding_Balance"].astype("float64")

    missing_ead = int(portfolio_ecl["EAD"].isna().sum())
    negative_ead = int((portfolio_ecl["EAD"] < 0).sum())
    ead_formula_exceptions = int((~np.isclose(
        portfolio_ecl["EAD"], portfolio_ecl["Outstanding_Balance"]
    )).sum())

    if missing_ead > 0 or negative_ead > 0 or ead_formula_exceptions > 0:
        raise ValueError("EAD derivation validation failed.")

    output_record_count = len(portfolio_ecl)
    records_processed = output_record_count

    print("=" * 80)
    print("EXPOSURE AT DEFAULT DERIVATION COMPLETED")
    print("=" * 80)
    print("EAD Rule: EAD = Outstanding_Balance")
    print("Missing EAD Values:", missing_ead)
    print("Negative EAD Values:", negative_ead)
    print("EAD Formula Exceptions:", ead_formula_exceptions)
    print("EAD Validation Status: PASS")

    display(portfolio_ecl[[
        "Customer_ID", "Loan_ID", "Scenario_Name", "Outstanding_Balance", "EAD"
    ]].head(15))

    logger.info("EAD derivation completed successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 3 failed during EAD derivation.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 08: ECL CALCULATION
BLOCK 3 : DERIVE EXPOSURE AT DEFAULT
START TIME : 2026-10-13 16:10:59
EXPOSURE AT DEFAULT DERIVATION COMPLETED
EAD Rule: EAD = Outstanding_Balance
Missing EAD Values: 0
Negative EAD Values: 0
EAD Formula Exceptions: 0
EAD Validation Status: PASS


,Customer_ID,Loan_ID,Scenario_Name,Outstanding_Balance,EAD
0,1000001,50000001,Baseline,1733362.19,1733362.19
1,1000001,50000001,Mild_Stress,1733362.19,1733362.19
2,1000001,50000001,Severe_Stress,1733362.19,1733362.19
3,1000002,50000002,Baseline,62244073.93,62244073.93
4,1000002,50000002,Mild_Stress,62244073.93,62244073.93
5,1000002,50000002,Severe_Stress,62244073.93,62244073.93
6,1000003,50000003,Baseline,303259.15,303259.15
7,1000003,50000003,Mild_Stress,303259.15,303259.15
8,1000003,50000003,Severe_Stress,303259.15,303259.15
9,1000004,50000004,Baseline,993246.23,993246.23


2026-10-13 16:10:59,955 | INFO | EAD derivation completed successfully.


INFO:08_ECL_Calculation:EAD derivation completed successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:10:59
DURATION (SECONDS) : 0.025


In [5]:
# =============================================================================
# CELL 05 - BLOCK 4: CALCULATE BASE AND STRESSED ECL
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 4 : CALCULATE BASE AND STRESSED ECL")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    portfolio_ecl["Base_ECL"] = (
        portfolio_ecl["EAD"] * portfolio_ecl["Base_PD"] * portfolio_ecl["Base_LGD"]
    )
    portfolio_ecl["Stressed_ECL"] = (
        portfolio_ecl["EAD"] * portfolio_ecl["Stressed_PD"] * portfolio_ecl["Stressed_LGD"]
    )
    portfolio_ecl["Incremental_Stress_Loss"] = (
        portfolio_ecl["Stressed_ECL"] - portfolio_ecl["Base_ECL"]
    )

    missing_ecl_values = int(portfolio_ecl[[
        "Base_ECL", "Stressed_ECL", "Incremental_Stress_Loss"
    ]].isna().sum().sum())
    negative_base_ecl = int((portfolio_ecl["Base_ECL"] < 0).sum())
    negative_stressed_ecl = int((portfolio_ecl["Stressed_ECL"] < 0).sum())

    base_formula_exceptions = int((~np.isclose(
        portfolio_ecl["Base_ECL"],
        portfolio_ecl["EAD"] * portfolio_ecl["Base_PD"] * portfolio_ecl["Base_LGD"]
    )).sum())
    stressed_formula_exceptions = int((~np.isclose(
        portfolio_ecl["Stressed_ECL"],
        portfolio_ecl["EAD"] * portfolio_ecl["Stressed_PD"] * portfolio_ecl["Stressed_LGD"]
    )).sum())
    incremental_formula_exceptions = int((~np.isclose(
        portfolio_ecl["Incremental_Stress_Loss"],
        portfolio_ecl["Stressed_ECL"] - portfolio_ecl["Base_ECL"]
    )).sum())

    if missing_ecl_values > 0 or negative_base_ecl > 0 or negative_stressed_ecl > 0:
        raise ValueError("Missing or negative ECL values were created.")
    if any([
        base_formula_exceptions,
        stressed_formula_exceptions,
        incremental_formula_exceptions
    ]):
        raise ValueError("One or more ECL formula validations failed.")

    output_record_count = len(portfolio_ecl)
    records_processed = output_record_count

    print("=" * 80)
    print("ECL CALCULATION COMPLETED")
    print("=" * 80)
    print("Missing ECL Values:", missing_ecl_values)
    print("Negative Base ECL Values:", negative_base_ecl)
    print("Negative Stressed ECL Values:", negative_stressed_ecl)
    print("Base ECL Formula Exceptions:", base_formula_exceptions)
    print("Stressed ECL Formula Exceptions:", stressed_formula_exceptions)
    print("Incremental Loss Formula Exceptions:", incremental_formula_exceptions)
    print("ECL Formula Validation Status: PASS")

    display(portfolio_ecl[[
        "Customer_ID", "Loan_ID", "Scenario_Name", "EAD", "Base_PD", "Base_LGD",
        "Base_ECL", "Stressed_PD", "Stressed_LGD", "Stressed_ECL",
        "Incremental_Stress_Loss"
    ]].head(20))

    logger.info("Base ECL, Stressed ECL, and incremental loss calculated successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 4 failed during ECL calculation.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 08: ECL CALCULATION
BLOCK 4 : CALCULATE BASE AND STRESSED ECL
START TIME : 2026-10-13 16:12:20
ECL CALCULATION COMPLETED
Missing ECL Values: 0
Negative Base ECL Values: 0
Negative Stressed ECL Values: 0
Base ECL Formula Exceptions: 0
Stressed ECL Formula Exceptions: 0
Incremental Loss Formula Exceptions: 0
ECL Formula Validation Status: PASS


,Customer_ID,Loan_ID,Scenario_Name,EAD,Base_PD,Base_LGD,Base_ECL,Stressed_PD,Stressed_LGD,Stressed_ECL,Incremental_Stress_Loss
0,1000001,50000001,Baseline,1733362.19,0.02,0.60,20800.346280,0.0200,0.6000,2.080035e+04,0.000000
1,1000001,50000001,Mild_Stress,1733362.19,0.02,0.60,20800.346280,0.0250,0.6600,2.860048e+04,7800.129855
2,1000001,50000001,Severe_Stress,1733362.19,0.02,0.60,20800.346280,0.0350,0.7500,4.550076e+04,24700.411208
3,1000002,50000002,Baseline,62244073.93,0.02,0.45,560196.665370,0.0200,0.4500,5.601967e+05,0.000000
4,1000002,50000002,Mild_Stress,62244073.93,0.02,0.45,560196.665370,0.0250,0.4950,7.702704e+05,210073.749514
5,1000002,50000002,Severe_Stress,62244073.93,0.02,0.45,560196.665370,0.0350,0.5625,1.225430e+06,665233.540127
6,1000003,50000003,Baseline,303259.15,0.10,0.85,25777.027750,0.1000,0.8500,2.577703e+04,0.000000
7,1000003,50000003,Mild_Stress,303259.15,0.10,0.85,25777.027750,0.1250,0.9350,3.544341e+04,9666.385406
8,1000003,50000003,Severe_Stress,303259.15,0.10,0.85,25777.027750,0.1750,1.0000,5.307035e+04,27293.323500
9,1000004,50000004,Baseline,993246.23,0.05,0.35,17381.809025,0.0500,0.3500,1.738181e+04,0.000000


2026-10-13 16:12:20,896 | INFO | Base ECL, Stressed ECL, and incremental loss calculated successfully.


INFO:08_ECL_Calculation:Base ECL, Stressed ECL, and incremental loss calculated successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:12:20
DURATION (SECONDS) : 0.05


In [6]:
# =============================================================================
# CELL 06 - BLOCK 5: GENERATE SCENARIO-WISE ECL SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 5 : GENERATE SCENARIO-WISE ECL SUMMARY")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    ecl_scenario_summary = (
        portfolio_ecl.groupby("Scenario_Name", sort=False)
        .agg(
            Record_Count=("Loan_ID", "size"),
            Base_ECL_Sum=("Base_ECL", "sum"),
            Base_ECL_Mean=("Base_ECL", "mean"),
            Base_ECL_Minimum=("Base_ECL", "min"),
            Base_ECL_Maximum=("Base_ECL", "max"),
            Stressed_ECL_Sum=("Stressed_ECL", "sum"),
            Stressed_ECL_Mean=("Stressed_ECL", "mean"),
            Stressed_ECL_Minimum=("Stressed_ECL", "min"),
            Stressed_ECL_Maximum=("Stressed_ECL", "max"),
            Incremental_Stress_Loss_Sum=("Incremental_Stress_Loss", "sum"),
            Incremental_Stress_Loss_Mean=("Incremental_Stress_Loss", "mean"),
            Incremental_Stress_Loss_Minimum=("Incremental_Stress_Loss", "min"),
            Incremental_Stress_Loss_Maximum=("Incremental_Stress_Loss", "max"),
        )
        .reset_index()
    )

    expected_scenario_counts = {
        "Baseline": 10000,
        "Mild_Stress": 10000,
        "Severe_Stress": 10000,
    }
    ecl_scenario_summary["Expected_Record_Count"] = (
        ecl_scenario_summary["Scenario_Name"].map(expected_scenario_counts)
    )
    ecl_scenario_summary["Status"] = np.where(
        ecl_scenario_summary["Record_Count"]
        == ecl_scenario_summary["Expected_Record_Count"],
        "PASS", "FAIL"
    )

    failed_scenario_counts = ecl_scenario_summary.loc[
        ecl_scenario_summary["Status"] != "PASS"
    ]
    if not failed_scenario_counts.empty:
        raise ValueError("Scenario-wise ECL record-count validation failed.")

    output_record_count = len(ecl_scenario_summary)
    records_processed = input_record_count

    print("=" * 80)
    print("SCENARIO-WISE ECL SUMMARY")
    print("=" * 80)
    display(ecl_scenario_summary.round(2))
    print("Scenario-wise ECL Summary Status: PASS")

    logger.info("Scenario-wise ECL summary generated successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 5 failed while generating scenario ECL summary.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)



MODULE 08: ECL CALCULATION
BLOCK 5 : GENERATE SCENARIO-WISE ECL SUMMARY
START TIME : 2026-10-13 16:14:41
SCENARIO-WISE ECL SUMMARY


,Scenario_Name,Record_Count,Base_ECL_Sum,Base_ECL_Mean,Base_ECL_Minimum,Base_ECL_Maximum,Stressed_ECL_Sum,Stressed_ECL_Mean,Stressed_ECL_Minimum,Stressed_ECL_Maximum,Incremental_Stress_Loss_Sum,Incremental_Stress_Loss_Mean,Incremental_Stress_Loss_Minimum,Incremental_Stress_Loss_Maximum,Expected_Record_Count,Status
0,Baseline,10000,9.299713e+08,92997.13,97.6,3345837.16,9.299713e+08,92997.13,97.60,3345837.16,0.000000e+00,0.00,0.00,0.00,10000,PASS
1,Mild_Stress,10000,9.299713e+08,92997.13,97.6,3345837.16,1.278711e+09,127871.05,134.20,4600526.10,3.487392e+08,34873.92,36.60,1254688.94,10000,PASS
2,Severe_Stress,10000,9.299713e+08,92997.13,97.6,3345837.16,2.031954e+09,203195.42,200.93,7319018.79,1.101983e+09,110198.29,103.34,3973181.63,10000,PASS


Scenario-wise ECL Summary Status: PASS
2026-10-13 16:14:41,281 | INFO | Scenario-wise ECL summary generated successfully.


INFO:08_ECL_Calculation:Scenario-wise ECL summary generated successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:14:41
DURATION (SECONDS) : 0.073


In [7]:
# =============================================================================
# CELL 07 - BLOCK 6: GENERATE RISK AND PRODUCT ECL SUMMARIES
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 6 : GENERATE RISK AND PRODUCT ECL SUMMARIES")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    ecl_risk_segment_summary = (
        portfolio_ecl.groupby(["Scenario_Name", "Risk_Segment"], sort=False)
        .agg(
            Record_Count=("Loan_ID", "size"),
            Stressed_ECL_Sum=("Stressed_ECL", "sum")
        )
        .reset_index()
    )

    ecl_product_summary = (
        portfolio_ecl.groupby(["Scenario_Name", "Product_Type"], sort=False)
        .agg(
            Record_Count=("Loan_ID", "size"),
            Stressed_ECL_Sum=("Stressed_ECL", "sum")
        )
        .reset_index()
    )

    risk_summary_record_total = int(ecl_risk_segment_summary["Record_Count"].sum())
    product_summary_record_total = int(ecl_product_summary["Record_Count"].sum())

    if risk_summary_record_total != input_record_count:
        raise ValueError("Risk-segment ECL summary does not reconcile to input records.")
    if product_summary_record_total != input_record_count:
        raise ValueError("Product ECL summary does not reconcile to input records.")

    output_record_count = len(ecl_risk_segment_summary) + len(ecl_product_summary)
    records_processed = input_record_count

    print("=" * 80)
    print("RISK SEGMENT AND PRODUCT ECL SUMMARIES")
    print("=" * 80)
    print("Risk Summary Record Total:", risk_summary_record_total)
    print("Product Summary Record Total:", product_summary_record_total)
    print("Risk Summary Reconciliation: PASS")
    print("Product Summary Reconciliation: PASS")
    print("\nScenario and Risk Segment ECL:")
    display(ecl_risk_segment_summary.round(2))
    print("\nScenario and Product ECL:")
    display(ecl_product_summary.round(2))

    logger.info("Risk-segment and product ECL summaries generated successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 6 failed while generating ECL summaries.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 08: ECL CALCULATION
BLOCK 6 : GENERATE RISK AND PRODUCT ECL SUMMARIES
START TIME : 2026-10-13 16:15:58
RISK SEGMENT AND PRODUCT ECL SUMMARIES
Risk Summary Record Total: 30000
Product Summary Record Total: 30000
Risk Summary Reconciliation: PASS
Product Summary Reconciliation: PASS

Scenario and Risk Segment ECL:


,Scenario_Name,Risk_Segment,Record_Count,Stressed_ECL_Sum
0,Baseline,Low Risk,2535,8.908115e+07
1,Mild_Stress,Low Risk,2535,1.224866e+08
2,Severe_Stress,Low Risk,2535,1.946506e+08
3,Baseline,High Risk,2485,4.214570e+08
4,Mild_Stress,High Risk,2485,5.795034e+08
5,Severe_Stress,High Risk,2485,9.208785e+08
6,Baseline,Medium Risk,4980,4.194331e+08
7,Mild_Stress,Medium Risk,4980,5.767205e+08
8,Severe_Stress,Medium Risk,4980,9.164251e+08



Scenario and Product ECL:


,Scenario_Name,Product_Type,Record_Count,Stressed_ECL_Sum
0,Baseline,Personal Loan,2117,7.134425e+07
1,Mild_Stress,Personal Loan,2117,9.809835e+07
2,Severe_Stress,Personal Loan,2117,1.560656e+08
3,Baseline,Corporate Loan,405,3.841704e+08
4,Mild_Stress,Corporate Loan,405,5.282344e+08
5,Severe_Stress,Corporate Loan,405,8.403729e+08
6,Baseline,Credit Card,1555,1.832473e+07
7,Mild_Stress,Credit Card,1555,2.519651e+07
8,Severe_Stress,Credit Card,1555,3.772739e+07
9,Baseline,Auto Loan,2088,7.183463e+07


2026-10-13 16:15:58,864 | INFO | Risk-segment and product ECL summaries generated successfully.


INFO:08_ECL_Calculation:Risk-segment and product ECL summaries generated successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 27
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:15:58
DURATION (SECONDS) : 0.065


In [8]:
# =============================================================================
# CELL 08 - BLOCK 7: IDENTIFY TOP 25 STRESSED ACCOUNTS
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 7 : IDENTIFY TOP 25 STRESSED ACCOUNTS")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    top_25_stressed_accounts = (
        portfolio_ecl.sort_values(
            by="Stressed_ECL",
            ascending=False,
            kind="stable"
        )
        .head(25)
        .reset_index(drop=True)
    )

    output_record_count = len(top_25_stressed_accounts)
    records_processed = input_record_count

    if output_record_count != 25:
        raise ValueError("Top stressed-account output does not contain 25 records.")
    if not top_25_stressed_accounts["Stressed_ECL"].is_monotonic_decreasing:
        raise ValueError("Top stressed accounts are not sorted by descending Stressed_ECL.")

    print("=" * 80)
    print("TOP 25 ACCOUNTS BY STRESSED ECL")
    print("=" * 80)
    display(top_25_stressed_accounts[[
        "Customer_ID", "Loan_ID", "Scenario_Name", "Risk_Segment",
        "Product_Type", "EAD", "Stressed_PD", "Stressed_LGD",
        "Stressed_ECL", "Incremental_Stress_Loss"
    ]].round(2))
    print("Top 25 Account Validation Status: PASS")

    logger.info("Top 25 stressed accounts identified successfully.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 7 failed while identifying top stressed accounts.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)



MODULE 08: ECL CALCULATION
BLOCK 7 : IDENTIFY TOP 25 STRESSED ACCOUNTS
START TIME : 2026-10-13 16:17:30
TOP 25 ACCOUNTS BY STRESSED ECL


,Customer_ID,Loan_ID,Scenario_Name,Risk_Segment,Product_Type,EAD,Stressed_PD,Stressed_LGD,Stressed_ECL,Incremental_Stress_Loss
0,1005503,50005503,Severe_Stress,High Risk,Corporate Loan,74351936.90,0.18,0.56,7319018.79,3973181.63
1,1000833,50000833,Severe_Stress,High Risk,Corporate Loan,72412384.54,0.18,0.56,7128094.10,3869536.80
2,1006773,50006773,Severe_Stress,High Risk,Corporate Loan,72344691.07,0.18,0.56,7121430.53,3865919.43
3,1004328,50004328,Severe_Stress,High Risk,Corporate Loan,70788686.34,0.18,0.56,6968261.31,3782770.43
4,1006883,50006883,Severe_Stress,High Risk,Corporate Loan,70447035.57,0.18,0.56,6934630.06,3764513.46
5,1009658,50009658,Severe_Stress,High Risk,Corporate Loan,68482049.28,0.18,0.56,6741201.73,3659509.51
6,1007000,50007000,Severe_Stress,High Risk,Corporate Loan,67866892.66,0.18,0.56,6680647.25,3626637.08
7,1001543,50001543,Severe_Stress,High Risk,Corporate Loan,67637225.55,0.18,0.56,6658039.39,3614364.24
8,1009705,50009705,Severe_Stress,High Risk,Corporate Loan,65388066.70,0.18,0.56,6436637.82,3494174.81
9,1006633,50006633,Severe_Stress,High Risk,Corporate Loan,65316108.72,0.18,0.56,6429554.45,3490329.56


Top 25 Account Validation Status: PASS
2026-10-13 16:17:30,898 | INFO | Top 25 stressed accounts identified successfully.


INFO:08_ECL_Calculation:Top 25 stressed accounts identified successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 25
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:17:30
DURATION (SECONDS) : 0.055


In [9]:
# =============================================================================
# CELL 09 - BLOCK 8: FINAL VALIDATION AND RECONCILIATION
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_lgd)
output_record_count = len(portfolio_ecl)
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 8 : FINAL VALIDATION AND RECONCILIATION")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    final_duplicate_keys = int(
        portfolio_ecl[["Loan_ID", "Scenario_ID"]].duplicated().sum()
    )
    final_missing_ead = int(portfolio_ecl["EAD"].isna().sum())
    final_missing_base_ecl = int(portfolio_ecl["Base_ECL"].isna().sum())
    final_missing_stressed_ecl = int(portfolio_ecl["Stressed_ECL"].isna().sum())
    final_missing_incremental_loss = int(
        portfolio_ecl["Incremental_Stress_Loss"].isna().sum()
    )

    validation_results = pd.DataFrame({
        "Validation_Check": [
            "Input and Output Record Count",
            "Expected Output Record Count",
            "Duplicate Loan-Scenario Keys",
            "Missing EAD",
            "Missing Base ECL",
            "Missing Stressed ECL",
            "Missing Incremental Stress Loss",
            "Negative EAD",
            "Negative Base ECL",
            "Negative Stressed ECL",
            "EAD Formula Exceptions",
            "Base ECL Formula Exceptions",
            "Stressed ECL Formula Exceptions",
            "Incremental Loss Formula Exceptions",
            "Risk Summary Record Total",
            "Product Summary Record Total",
            "Top Stressed Account Count",
        ],
        "Expected_Result": [
            input_record_count, 30000, 0, 0, 0, 0, 0, 0, 0, 0,
            0, 0, 0, 0, 30000, 30000, 25
        ],
        "Actual_Result": [
            output_record_count, output_record_count, final_duplicate_keys,
            final_missing_ead, final_missing_base_ecl, final_missing_stressed_ecl,
            final_missing_incremental_loss, negative_ead, negative_base_ecl,
            negative_stressed_ecl, ead_formula_exceptions,
            base_formula_exceptions, stressed_formula_exceptions,
            incremental_formula_exceptions, risk_summary_record_total,
            product_summary_record_total, len(top_25_stressed_accounts)
        ],
    })
    validation_results["Status"] = np.where(
        validation_results["Expected_Result"].astype(str)
        == validation_results["Actual_Result"].astype(str),
        "PASS", "FAIL"
    )

    ecl_calculation_summary = pd.DataFrame({
        "Metric": [
            "Input Records", "Output Records", "Input Columns", "Output Columns",
            "Total EAD", "Total Base ECL", "Total Stressed ECL",
            "Total Incremental Stress Loss", "Minimum Base ECL",
            "Maximum Base ECL", "Minimum Stressed ECL",
            "Maximum Stressed ECL", "Processing Timestamp"
        ],
        "Result": [
            input_record_count, output_record_count, input_column_count,
            len(portfolio_ecl.columns), float(portfolio_ecl["EAD"].sum()),
            float(portfolio_ecl["Base_ECL"].sum()),
            float(portfolio_ecl["Stressed_ECL"].sum()),
            float(portfolio_ecl["Incremental_Stress_Loss"].sum()),
            float(portfolio_ecl["Base_ECL"].min()),
            float(portfolio_ecl["Base_ECL"].max()),
            float(portfolio_ecl["Stressed_ECL"].min()),
            float(portfolio_ecl["Stressed_ECL"].max()),
            datetime.now()
        ],
    })

    failed_validations = validation_results.loc[
        validation_results["Status"] != "PASS"
    ]
    if not failed_validations.empty:
        raise ValueError("One or more Module 08 validation checks failed.")

    records_processed = output_record_count

    print("=" * 80)
    print("FINAL ECL CALCULATION VALIDATION")
    print("=" * 80)
    display(validation_results)
    print("\nECL Calculation Module Summary:")
    display(ecl_calculation_summary)
    print("Overall Module 08 Validation Status: PASS")

    logger.info("All Module 08 validation checks passed.")
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 8 failed during final validation.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)




MODULE 08: ECL CALCULATION
BLOCK 8 : FINAL VALIDATION AND RECONCILIATION
START TIME : 2026-10-13 16:19:16
FINAL ECL CALCULATION VALIDATION


,Validation_Check,Expected_Result,Actual_Result,Status
0,Input and Output Record Count,30000,30000,PASS
1,Expected Output Record Count,30000,30000,PASS
2,Duplicate Loan-Scenario Keys,0,0,PASS
3,Missing EAD,0,0,PASS
4,Missing Base ECL,0,0,PASS
5,Missing Stressed ECL,0,0,PASS
6,Missing Incremental Stress Loss,0,0,PASS
7,Negative EAD,0,0,PASS
8,Negative Base ECL,0,0,PASS
9,Negative Stressed ECL,0,0,PASS



ECL Calculation Module Summary:


,Metric,Result
0,Input Records,30000
1,Output Records,30000
2,Input Columns,38
3,Output Columns,42
4,Total EAD,142745720390.970001
5,Total Base ECL,2789913837.14619
6,Total Stressed ECL,4240635998.746957
7,Total Incremental Stress Loss,1450722161.600767
8,Minimum Base ECL,97.59649
9,Maximum Base ECL,3345837.1605


Overall Module 08 Validation Status: PASS
2026-10-13 16:19:16,792 | INFO | All Module 08 validation checks passed.


INFO:08_ECL_Calculation:All Module 08 validation checks passed.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:19:16
DURATION (SECONDS) : 0.057


In [10]:
# =============================================================================
# CELL 10 - BLOCK 9: SAVE OUTPUTS, VERIFY FILES, AND CLOSE MODULE
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(portfolio_ecl)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 08: ECL CALCULATION")
print("BLOCK 9 : SAVE OUTPUTS AND VERIFY FILES")
print("START TIME :", block_start_time.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

try:
    portfolio_ecl.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m-%d")
    ecl_calculation_summary.to_csv(SUMMARY_FILE, index=False)
    validation_results.to_csv(VALIDATION_FILE, index=False)
    ecl_scenario_summary.to_csv(SCENARIO_SUMMARY_FILE, index=False)
    ecl_risk_segment_summary.to_csv(RISK_SUMMARY_FILE, index=False)
    ecl_product_summary.to_csv(PRODUCT_SUMMARY_FILE, index=False)
    top_25_stressed_accounts.to_csv(TOP_ACCOUNTS_FILE, index=False)
    portfolio_ecl.head(25).to_csv(SAMPLE_FILE, index=False, date_format="%Y-%m-%d")

    logger.info("ECL portfolio and validation evidence saved successfully.")

    expected_output_files = [
        OUTPUT_FILE, SUMMARY_FILE, VALIDATION_FILE, SCENARIO_SUMMARY_FILE,
        RISK_SUMMARY_FILE, PRODUCT_SUMMARY_FILE, TOP_ACCOUNTS_FILE,
        SAMPLE_FILE, LOG_FILE
    ]
    output_verification = pd.DataFrame({
        "File_Name": [file_path.name for file_path in expected_output_files],
        "Exists": [file_path.exists() for file_path in expected_output_files],
        "Size_Bytes": [
            file_path.stat().st_size if file_path.exists() else 0
            for file_path in expected_output_files
        ],
        "Full_Path": [str(file_path) for file_path in expected_output_files],
    })

    if not output_verification["Exists"].all():
        raise FileNotFoundError("One or more Module 08 output files were not created.")

    saved_output = pd.read_csv(OUTPUT_FILE, low_memory=False)
    output_record_count = len(saved_output)
    if output_record_count != 30000:
        raise ValueError(
            "Saved ECL portfolio does not contain the expected 30,000 records."
        )

    saved_formula_exceptions = int((~np.isclose(
        saved_output["Stressed_ECL"],
        saved_output["EAD"] * saved_output["Stressed_PD"] * saved_output["Stressed_LGD"]
    )).sum())
    if saved_formula_exceptions > 0:
        raise ValueError("Saved ECL output failed formula verification.")

    records_processed = output_record_count
    block_status = "SUCCESS"
except Exception:
    block_status = "FAILED"
    logger.exception("Block 9 failed while saving or verifying outputs.")
    raise
finally:
    block_end_time = datetime.now()
    block_duration_seconds = round((block_end_time - block_start_time).total_seconds(), 3)
    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print("END TIME           :", block_end_time.strftime("%Y-%m-%d %H:%M:%S"))
    print("DURATION (SECONDS) :", block_duration_seconds)
    print("=" * 80)

module_end_timestamp = datetime.now()
module_duration_seconds = round(
    (module_end_timestamp - module_start_timestamp).total_seconds(), 3
)

print("=" * 80)
print("PYTHON V2 - MODULE 08 COMPLETED")
print("ECL CALCULATION EXECUTED SUCCESSFULLY")
print("=" * 80)
print("Input Records:", input_record_count)
print("ECL Records:", output_record_count)
print("ECL Output:", OUTPUT_FILE)
print("Validation Summary:", SUMMARY_FILE)
print("Validation Evidence:", VALIDATION_FILE)
print("Scenario ECL Summary:", SCENARIO_SUMMARY_FILE)
print("Risk Segment ECL Summary:", RISK_SUMMARY_FILE)
print("Product ECL Summary:", PRODUCT_SUMMARY_FILE)
print("Top 25 Stressed Accounts:", TOP_ACCOUNTS_FILE)
print("Output Sample:", SAMPLE_FILE)
print("Execution Log:", LOG_FILE)
print("\nOutput File Verification:")
display(output_verification)
print("=" * 80)
print("MODULE STATUS             : SUCCESS")
print("MODULE START              :", module_start_timestamp.strftime("%Y-%m-%d %H:%M:%S"))
print("MODULE END                :", module_end_timestamp.strftime("%Y-%m-%d %H:%M:%S"))
print("TOTAL DURATION (SECONDS)  :", module_duration_seconds)
print("TOTAL RECORDS PROCESSED   :", output_record_count)
print("=" * 80)
print("All Module 08 output files verified successfully.")

logger.info(
    "Module 08 completed successfully in %s seconds.",
    module_duration_seconds
)


MODULE 08: ECL CALCULATION
BLOCK 9 : SAVE OUTPUTS AND VERIFY FILES
START TIME : 2026-10-13 16:20:56
2026-10-13 16:20:58,073 | INFO | ECL portfolio and validation evidence saved successfully.


INFO:08_ECL_Calculation:ECL portfolio and validation evidence saved successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 16:20:58
DURATION (SECONDS) : 2.182
PYTHON V2 - MODULE 08 COMPLETED
ECL CALCULATION EXECUTED SUCCESSFULLY
Input Records: 30000
ECL Records: 30000
ECL Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_ecl.csv
Validation Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/08_ecl_calculation_summary.csv
Validation Evidence: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/08_ecl_calculation_validation.csv
Scenario ECL Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/08_ecl_scenario_summary.csv
Risk Segment ECL Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/08_ecl_risk_segment_summary.csv
Product ECL Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/08_ecl_product_summary.csv
Top 25 Stressed Accoun

,File_Name,Exists,Size_Bytes,Full_Path
0,portfolio_ecl.csv,True,9293904,/content/drive/MyDrive/Banking_Stress_Testing_...
1,08_ecl_calculation_summary.csv,True,414,/content/drive/MyDrive/Banking_Stress_Testing_...
2,08_ecl_calculation_validation.csv,True,665,/content/drive/MyDrive/Banking_Stress_Testing_...
3,08_ecl_scenario_summary.csv,True,955,/content/drive/MyDrive/Banking_Stress_Testing_...
4,08_ecl_risk_segment_summary.csv,True,455,/content/drive/MyDrive/Banking_Stress_Testing_...
5,08_ecl_product_summary.csv,True,877,/content/drive/MyDrive/Banking_Stress_Testing_...
6,08_top_25_stressed_accounts.csv,True,8927,/content/drive/MyDrive/Banking_Stress_Testing_...
7,08_ecl_calculation_sample.csv,True,8386,/content/drive/MyDrive/Banking_Stress_Testing_...
8,08_ECL_Calculation.log,True,792,/content/drive/MyDrive/Banking_Stress_Testing_...


MODULE STATUS             : SUCCESS
MODULE START              : 2026-10-13 16:04:16
MODULE END                : 2026-10-13 16:20:58
TOTAL DURATION (SECONDS)  : 1002.243
TOTAL RECORDS PROCESSED   : 30000
All Module 08 output files verified successfully.
2026-10-13 16:20:58,416 | INFO | Module 08 completed successfully in 1002.243 seconds.


INFO:08_ECL_Calculation:Module 08 completed successfully in 1002.243 seconds.
